<h1>YodAI: An AI-based Search Tool</h1>

<p>We have been tasked with the following things until the IRD doc has been approved. As such, we will work on the pipeline development based on the datasets and metadata we currently have.</p>

<h3>To-do</h3>

<ol>
<li><b>Begin Pipeline Development Using Metadata</b>

Start building the framework for:
<ul>
<li>Data ingestion</li>
<li>Cleaning</li>
<li>Querying</li>
<li>Modeling</li>
<li>Using metadata or synthetic data.</li>
</ul>
</li>


<li><b>Design Data Cleaning Pipeline</b>

Prepare scripts to:
<ul>
<li>Standardize variables</li>
<li>Handle missing data</li>
<li>Normalize categorical fields</li>
<li>Structure patient-visit relationships.</li>
</ul>
</li>


<li><b>Build Queryable Database Structure</b>

Design the structure for:
<ul>
<li>Patient table
<li>Visit table
<li>Labs table
<li>Complications table.</li>
</ul>
</li>


<li><b>Create Synthetic Dataset for Testing</b>

Generate simulated data that matches the metadata schema so the team can:
<ul>
<li>Test pipelines</li>
<li>Debug models</li>
<li>Develop queries.</li>
</ul>
</li>


<li><b>Define Modeling Goals</b>

Prepare analytic tasks such as:
<ul>
<li>Predict diabetes complications</li>
<li>Identify risk clusters</li>
<li>Analyze treatment patterns.</li>
</ul>
</li>


<li><b>Literature Review</b>

Students should begin:
<ul>
<li>Literature review on diabetes registries</li>
<li>ML models for diabetes outcomes</li>
<li>Clinical risk prediction.</li>
</ul>
</li>


<li><b>Prepare for Data Integration</b>

Once data arrives:
<ul>
<li>Plug dataset into pipeline</li>
<li>Run cleaning</li>
<li>Conduct exploratory analysis.</li>
</ul>
</li>
</ol>

<h2>0. Importing libraries</h2>

In [2]:
import pandas as pd
import numpy as np
import re
import os

<h2>1. Exploring dataset</h2>

In [3]:
df_metadata = pd.read_csv('dmc_data_structure_metadata.csv')
df_sample = pd.read_excel('dmc_data_sample.xlsx')

print('---------------------------- METADATA ----------------------------\n', df_metadata.head())
print('\n----------------------------- SAMPLE -----------------------------\n', df_sample.head())

---------------------------- METADATA ----------------------------
   Column_Name  Data_Type  Missing_Count  Missing_Percentage  Unique_Values
0         Pin  character              0                0.00          88132
1   FirstName  character           9983                2.57           8307
2    LastName  character           9983                2.57           5535
3  FatherName  character           9984                2.57          14981
4    Birthday  character           9983                2.57             16

----------------------------- SAMPLE -----------------------------
       Pin  Gender RegistrationDate MartialStatus  VisitDate  Expr2 Expr3  id  \
0  310351  Female       2015-07-15       Married 2015-07-15    NaN    DM   2   
1  310351  Female       2015-07-15       Married 2015-07-15    NaN    DM   3   
2  310358    Male       2015-07-16           NaN 2015-07-16    NaN    DM   4   
3  310360  Female       2015-07-16           NaN 2015-07-16    NaN    DM   8   
4  310364    

In [4]:
metadata_columns = df_metadata['Column_Name'].tolist()
sample_columns = df_sample.keys().tolist()

print(len(metadata_columns), metadata_columns)
print(len(sample_columns), sample_columns)

199 ['Pin', 'FirstName', 'LastName', 'FatherName', 'Birthday', 'Gender', 'RegistrationDate', 'PhoneNo1', 'MartialStatus', 'VisitDate', 'DoctorName', 'Expr2', 'Expr3', 'id...14', 'DiagnosticScore', 'DiagnosticStatus', 'EtiologicalScore', 'EtiologicalStatus', 'FunctionalStatus', 'FunctionalStatus2', 'Complications', 'Referalreasonagos', 'ReasonForReferals', 'InsulinR', 'InsulinRFullDose', 'InsulinN', 'InsulinNFullDose', 'Insulin7030', 'Insulin7030FullDose', 'ACarboseFullDose', 'ACarbose', 'NextVisit', 'PatientTreatmentPlan', 'CurrentSuggestedInsulin', 'CurrentSuggestedOHG', 'DietCalories', 'Instruction2', 'MetforminDose', 'MetforminFullDose', 'SitagliptinDose', 'SitagliptinFullDose', 'GlimiprideDose', 'GlimiprideFullDose', 'PioglitazoneDose', 'PioglitazoneFullDose', 'GITDrug1', 'GITDrug1FullDose', 'GITDrug2', 'GITDrug2FullDose', 'GITDrug3', 'GITDrug3FullDose', 'NeuropathyPainKiller1', 'NeuropathyPainKiller1Dose', 'NeuropathyPainKiller2', 'NeuropathyPainKiller2Dose', 'NeuropathyPainKiller

<h3>Comparing the two datasets we currently have</h3>

In [5]:
# Comparing sample columns against metadata columns
missing_metadata_columns = []
for column in metadata_columns:
    if column not in sample_columns:
        missing_metadata_columns.append(column)

print('Missing columns in sample data:', missing_metadata_columns)
print('Total missing columns:', len(missing_metadata_columns))

Missing columns in sample data: ['FirstName', 'LastName', 'FatherName', 'Birthday', 'PhoneNo1', 'DoctorName', 'id...14', 'RefferedBy', 'Consultant', 'PlannedBy', 'id...159', 'seq']
Total missing columns: 12


In [6]:
# Comparing metadata columns against sample columns
missing_sample_columns = []
for column in sample_columns:
    if column not in metadata_columns:
        missing_sample_columns.append(column)

print('Missing columns in sample data:', missing_sample_columns)
print('Total missing columns:', len(missing_sample_columns))

Missing columns in sample data: ['id', 'id.1']
Total missing columns: 2


<h2>Information about current datasets</h2>

We currently have two datasets - DMC metadata, and Sample DMC data. Below is the information about each of them:

<ul>
<li>
<b>DMC metadata</b> (dmc_data_structure_metadata.csv)

This metadata for the original dataset has 199 values with 4 feature traits. It contains information about the column names of the original dataset, and some very basic information as show below:

<ul>
<li><b>Data_Type:</b> All the elements are "character" except "Expr4" and "seq". This represents the original dataset having primarily character-based features (other than these two features).</li>
<li><b>Missing_Count:</b> Shows missing values in each column</li>
<li><b>Missing_Percentage:</b> Shows what percentage of the entire column is missing</li>
<li><b>Unique_Values:</b> Shows number of unique elements in each column</li>
</ul>
</li>


<li>
<b>Sample DMC data</b> (dmc_data_sample.xlsx)

This dataset contains a total of 29 unique values, over 189 features. Roughly half the features are either completely NULL values or completely duplicated values - only a few of them have 1-2 non-NULL/non-duplicate values.

Other than that, this dataset contains 2 extra columns "id" and "Id.1". Moreover, it has 10 features absent that are otherwise present in the DMC metadata. Note that since this is just a sample dataset, you don't need to merge them.
</li>
</ul>

<hr>

<h2>2. Data cleaning</h2>

Given the overall structure of both the datasets, I would recommend doing the following for clarity:

<ol>
<li>Standardize all non-numerical feature values</li>
<li>Remove features that have only-NULL or a single value for all elements</li>
<li>Alter column values appropriately (set null to 0 or "N/A")</li>
<li>See which features are purely unique to set the primary key accordingly. You can start with 'Expr4' and 'seq' first since they're the only numerical columns</li>
</ol>
